In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


# Load data
df = pd.read_csv('/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv')
# Drop the 'Class' (labels) 
# We want the model to learn features WITHOUT knowing what is fraud.
# Consider features [V1-V28, Amount], total 29 features
X = df.drop(['Class', 'Time'], axis=1).values 

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split for training
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

In [ ]:
class SSL_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 29 features -> 384-d latent space
        self.encoder = nn.Sequential(
            nn.Linear(29, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 384) # Our final embedding size for Supabase
        )
        # Decoder: Reconstruct back to 29 features
        self.decoder = nn.Sequential(
            nn.Linear(384, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 29)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

model = SSL_Encoder().cuda() # Move to GPU
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 20
batch_size = 1024
train_loader = DataLoader(TensorDataset(X_train), batch_size=batch_size, shuffle=True)

print("Starting SSL Training...")
for epoch in range(epochs):
    for data in train_loader:
        img = data[0].cuda()
        # Forward
        encoded, decoded = model(img)
        loss = criterion(decoded, img)
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Save the model
torch.save(model.state_dict(), 'ssl_encoder.pth')
print("Model Saved!")